# The Fine-Tuning Landscape: From Full Fine-Tuning to Parameter-Efficient Methods

Fine-tuning a pre-trained language model on task-specific data is one of the most powerful techniques in applied NLP. But the cost structure changed dramatically as models scaled from millions to billions of parameters.

## Full Fine-Tuning: When It Made Sense

In the BERT era (2018-2021), full fine-tuning was standard practice. BERT-base has 110M parameters — small enough to fit on a single GPU with a learning rate scheduler. You initialized from the pre-trained checkpoint, added a task-specific head, and updated all parameters for a few epochs.

This worked because:
- Models were small enough for full gradient computation
- Task datasets were often large (tens of thousands of examples)
- Storage was cheap relative to model size
- The gap between pre-trained and fine-tuned representations was small enough that updating all weights was safe

Full fine-tuning produced state-of-the-art results on GLUE, SQuAD, and classification benchmarks through 2022.

## Why Full Fine-Tuning Stopped Scaling

As models grew to 7B, 13B, 70B parameters, full fine-tuning became impractical for most practitioners:

**Memory cost**: fine-tuning a 7B model requires storing parameters (28GB in fp32), gradients (28GB), and optimizer state (56GB for Adam). Total: ~112GB just for Adam states — more than 4 A100s.

**Storage cost**: each fine-tuned variant requires a separate full model checkpoint. Ten fine-tuned versions of Llama-70B = 4TB of storage.

**Catastrophic forgetting**: aggressive fine-tuning on narrow task distributions degrades the model's general capabilities, even when the fine-tuning data is high quality.

These pressures drove the development of parameter-efficient fine-tuning (PEFT) methods, which adapt the model by modifying only a small fraction of parameters.

In [ ]:
# Demonstrate the memory math for full fine-tuning vs LoRA
def memory_breakdown(n_params_billions: float, method: str = 'full_ft') -> dict:
    n = n_params_billions * 1e9
    bytes_per_param_fp32 = 4
    bytes_per_param_bf16 = 2
    if method == 'full_ft':
        params_mem = n * bytes_per_param_bf16  # model in bf16
        grad_mem = n * bytes_per_param_fp32    # gradients in fp32
        # Adam: first + second moment, both fp32
        optimizer_mem = 2 * n * bytes_per_param_fp32
        total = params_mem + grad_mem + optimizer_mem
        trainable_params = n
    elif method == 'lora_r8':
        # LoRA rank 8, targeting q and v projections (~1% of params)
        lora_fraction = 0.01
        lora_params = n * lora_fraction
        params_mem = n * bytes_per_param_bf16       # frozen base in bf16
        grad_mem = lora_params * bytes_per_param_fp32
        optimizer_mem = 2 * lora_params * bytes_per_param_fp32
        total = params_mem + grad_mem + optimizer_mem
        trainable_params = lora_params
    elif method == 'qlora_4bit':
        # 4-bit quantized base + LoRA adapters
        lora_fraction = 0.01
        lora_params = n * lora_fraction
        params_mem = n * 0.5                        # 4-bit = 0.5 bytes/param
        grad_mem = lora_params * bytes_per_param_fp32
        optimizer_mem = 2 * lora_params * bytes_per_param_fp32
        total = params_mem + grad_mem + optimizer_mem
        trainable_params = lora_params
    return {
        'method': method,
        'model_size_B': n_params_billions,
        'trainable_params_M': round(trainable_params/1e6, 1),
        'trainable_pct': round(trainable_params/n*100, 2),
        'total_mem_GB': round(total/1e9, 1),
    }

print(f'{'Method':<15} {'Trainable':>15} {'% Params':>10} {'Total Mem':>12}')
print('-' * 55)
for model_b in [7, 13, 70]:
    print(f'--- {model_b}B parameters ---')
    for method in ['full_ft', 'lora_r8', 'qlora_4bit']:
        r = memory_breakdown(model_b, method)
        print(f"  {r['method']:<13} {r['trainable_params_M']:>12.0f}M {r['trainable_pct']:>9.2f}% {r['total_mem_GB']:>10.1f}GB")

## Parameter-Efficient Fine-Tuning Taxonomy

The PEFT landscape organized into three families:

**Adapter methods** (Houlsby et al. 2019): insert small feed-forward modules between transformer layers. The base model is frozen; only adapter weights are trained. Effective but adds inference latency since the adapter layers run at every forward pass.

**Soft prompt methods** (Lester et al. 2021 — Prompt Tuning; Li & Liang 2021 — Prefix Tuning): prepend a small number of trainable token embeddings to the input. No added inference latency. Works surprisingly well at scale but underperforms at smaller model sizes.

**Low-rank decomposition methods** (LoRA, AdaLoRA): decompose weight updates into low-rank matrices. Zero inference overhead when merged back into the base weights. This family won: LoRA became the dominant PEFT method by 2023.

The convergence on LoRA-family methods reflects the key practical advantage: after training, the LoRA weights can be merged into the base model, so inference is identical to a full fine-tune with no overhead.

In [ ]:
# Visualize the PEFT landscape
methods = [
    {'name': 'Full Fine-Tune', 'trainable_pct': 100, 'inference_overhead': 0, 'memory_gb_7b': 112},
    {'name': 'Adapters', 'trainable_pct': 3, 'inference_overhead': 8, 'memory_gb_7b': 16},
    {'name': 'Prefix Tuning', 'trainable_pct': 0.1, 'inference_overhead': 2, 'memory_gb_7b': 15},
    {'name': 'Prompt Tuning', 'trainable_pct': 0.01, 'inference_overhead': 0, 'memory_gb_7b': 14},
    {'name': 'LoRA (r=8)', 'trainable_pct': 1, 'inference_overhead': 0, 'memory_gb_7b': 16},
    {'name': 'QLoRA (4-bit)', 'trainable_pct': 1, 'inference_overhead': 0, 'memory_gb_7b': 6},
]

print(f'{'Method':<20} {'Trainable %':>12} {'Infer Overhead':>15} {'Mem 7B (GB)':>13}')
print('-' * 65)
for m in methods:
    print(f"{m['name']:<20} {m['trainable_pct']:>11.2f}% {m['inference_overhead']:>13}% {m['memory_gb_7b']:>12}GB")

## When to Use Each Method

**Full fine-tuning**: only when you control the training infrastructure, have a large high-quality dataset (100K+ examples), and need maximum task performance. Research labs, not practitioners.

**LoRA**: the default choice for most fine-tuning work. Rank 8-64 covers most use cases. Start with r=16 targeting Q, K, V, and output projections.

**QLoRA**: when you need to fine-tune a large model (13B+) on a single GPU. The quantization step introduces slight quality degradation, but QLoRA fine-tuned 13B models often outperform full fine-tuned 7B models.

**Prompt/Prefix tuning**: when you need to maintain many task variants of the same base model and cannot afford separate LoRA adapters. Works well only at 10B+ parameter scale.